# Hitting the Vibe：算法复现（2026 年 11 月）

**这是什么：** 2024 年春季 NYU *Data Analysis in the Music Industry* 课程小组项目（Fred Xiang、Owen Tynes、Sam Richter、Gerrit Wendt）的算法，按期末论文里写的方法步骤重新实现一遍。目的是弄清楚每一步在做什么，并用数据检验论文里的两个结论。

**和原项目的区别（要如实说明）：**
- Spotify 从 2024 年底起不再给新应用开放 audio features 接口，所以这里用公开数据集 TidyTuesday `spotify_songs`（约 2.8 万首歌，带 Spotify 音频特征、歌单名和歌单流派）代替实时 API。
- 原项目用 Billboard Hot 200 加全球历史前 100 共 300 首歌组成歌曲池；这里用数据集里最热门的 300 首代替。

**结构：** 第 1–4 步复现原算法；第 5 步检验原论文的两个结论；最后是这次复现发现的问题。

In [1]:
import pandas as pd, numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

raw = pd.read_csv("spotify_songs.csv")
# A song can appear in several playlists: keep one row per song,
# but remember whether ANY of its playlists has "party" in the name (used in step 5).
in_party = raw.groupby("track_id").playlist_name.apply(lambda s: s.str.contains("party", case=False).any())
songs = raw.drop_duplicates("track_id").set_index("track_id")
songs["in_party_playlist"] = in_party
print(f"{len(raw):,} rows -> {len(songs):,} unique songs")

32,833 rows -> 28,356 unique songs


## 第 1 步：生成 10 位「客人」的歌单

原论文：把 300 首热门歌放在一起，用随机数为 10 位客人各抽 25 首，当作他们的 Spotify Top 25。

In [2]:
# Many songs tie on popularity at the cutoff (57 songs share the 300th value), so break ties by
# track_id with a stable sort; otherwise the pool, and every result after it, can differ between machines.
pool = (songs.reset_index()
             .sort_values(["track_popularity", "track_id"], ascending=[False, True], kind="mergesort")
             .set_index("track_id").head(300))
rng = np.random.default_rng(42)
guests = {f"Guest {i+1}": pool.sample(25, random_state=int(rng.integers(1_000_000))) for i in range(10)}
print({k: len(v) for k, v in guests.items()})

{'Guest 1': 25, 'Guest 2': 25, 'Guest 3': 25, 'Guest 4': 25, 'Guest 5': 25, 'Guest 6': 25, 'Guest 7': 25, 'Guest 8': 25, 'Guest 9': 25, 'Guest 10': 25}


## 第 2 步：给每首歌打「派对分」，每位客人取前 10

原论文的公式：**派对分 = danceability + energy − acousticness**。没有用固定阈值，而是每位客人取分数最高的 10 首。

In [3]:
songs["party_score"] = songs.danceability + songs.energy - songs.acousticness
for g in guests.values():
    g["party_score"] = g.danceability + g.energy - g.acousticness
candidates = pd.concat([g.nlargest(10, "party_score") for g in guests.values()])
candidates = candidates[~candidates.index.duplicated()]   # same hit can be in several guests' lists
print(len(candidates), "candidate songs after removing duplicates")

77 candidate songs after removing duplicates


## 第 3 步：用余弦相似度选出最「合群」的 25 首

原论文：用 6 个已经在 0–1 之间的特征（danceability、energy、speechiness、instrumentalness、acousticness、valence）组成矩阵，算每两首歌的余弦相似度。每首歌和其他所有歌的平均相似度越高，说明它越「合群」。取平均相似度最高的 25 首。

In [4]:
FEATURES = ["danceability", "energy", "speechiness", "instrumentalness", "acousticness", "valence"]

def build_mix(cands, X, n=25):
    sim = cosine_similarity(X)
    np.fill_diagonal(sim, np.nan)          # ignore each song's similarity with itself
    out = cands.copy()
    out["mean_sim"] = np.nanmean(sim, axis=1)
    return out.nlargest(n, "mean_sim")

mix = build_mix(candidates, candidates[FEATURES])
mix[["track_name", "track_artist", "playlist_genre", "party_score", "mean_sim"]].round(3)

,track_name,track_artist,playlist_genre,party_score,mean_sim
track_id,,,,,
7rknFhrVDyxzTJkPSf7LoW,No Me Conoce - Remix,Jhay Cortez,rap,1.440,0.976
7ef4DlsgrMEH11cDZd32M6,One Kiss (with Dua Lipa),Calvin Harris,pop,1.616,0.975
58q2HKrzhC3ozto2nDdN4z,I Like It,Cardi B,pop,1.443,0.975
0pqnGHJpmpxLKifKRmU6WP,Believer,Imagine Dragons,latin,1.494,0.975
2BgEsaKNfHUdlh97KmvFyo,2002,Anne-Marie,pop,1.343,0.974
059bcIhyc2SBwm6sw2AZzd,Te Vi,Piso 21,latin,1.609,0.974
6WrI0LAC5M1Rw2MnX2ZvEg,Don't Start Now,Dua Lipa,pop,1.575,0.973
4w8niZpiMy6qz1mntFA5uM,"Taki Taki (with Selena Gomez, Ozuna & Cardi B)",DJ Snake,pop,1.486,0.973
25sgk305KZfyuqVBQIahim,Sweet but Psycho,Ava Max,pop,1.354,0.973


## 第 4 步：结果概况

In [5]:
print("Genres in the mix:", mix.playlist_genre.value_counts().to_dict())
print("Mean-similarity range of all candidates:",
      round(build_mix(candidates, candidates[FEATURES], n=len(candidates)).mean_sim.min(), 3), "to",
      round(mix.mean_sim.max(), 3))

Genres in the mix: {'pop': 15, 'rap': 5, 'latin': 5}
Mean-similarity range of all candidates: 0.885 to 0.976


## 第 5 步：检验原论文的两个结论

**结论 A：只靠音频特征，无法设一个阈值把「派对歌」挑出来。**

检验方法：把「出现在歌单名含 party 的歌单里」当作派对歌的粗略标签，看各个分数能多好地区分派对歌和其他歌。指标是 **AUC**：0.5 等于瞎猜，1.0 是完美区分。

In [6]:
y = songs.in_party_playlist
print(f"Party-playlist songs: {y.sum()} of {len(y):,} ({y.mean():.1%})")
for col in ["danceability", "energy", "party_score"]:
    print(f"AUC of {col:12s}: {roc_auc_score(y, songs[col]):.3f}")

Party-playlist songs: 670 of 28,356 (2.4%)
AUC of danceability: 0.481
AUC of energy      : 0.583


AUC of party_score : 0.596


**结论 B：音频特征不能很好地预测流派。**

检验方法：用 6 个特征训练随机森林预测 6 种流派（pop、rap、rock、latin、r&b、edm），5 折交叉验证，和「全部猜最多的那一类」比较。

In [7]:
sample = songs.sample(8000, random_state=0)
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0)
acc = cross_val_score(rf, sample[FEATURES], sample.playlist_genre, cv=5).mean()
baseline = sample.playlist_genre.value_counts(normalize=True).max()
print(f"Genre accuracy from audio features: {acc:.1%}  (majority-class baseline: {baseline:.1%})")

Genre accuracy from audio features: 47.2%  (majority-class baseline: 19.0%)


## 第 6 步：复现中发现的问题——结果对预处理很敏感

第 4 步里，所有候选歌的平均相似度都在 0.88 以上。原因：6 个特征都是正数，任何两首歌的特征向量方向都差不多，余弦相似度天然偏高。这时排序主要由很小的差别决定。

另一种常见做法是先把每个特征**标准化**（减均值、除标准差）再算余弦相似度，比较的是「这首歌在哪些方面高于或低于平均」。下面看看换一种预处理，歌单会变多少。

In [8]:
X_std = StandardScaler().fit_transform(candidates[FEATURES])
raw_all = build_mix(candidates, candidates[FEATURES], n=len(candidates)).mean_sim
std_all = build_mix(candidates, X_std, n=len(candidates)).mean_sim
print(f"Raw features:          mean_sim spread = {raw_all.max()-raw_all.min():.3f}")
print(f"Standardized features: mean_sim spread = {std_all.max()-std_all.min():.3f}")
mix_std = build_mix(candidates, X_std)
print("Songs that change between the two mixes:", len(set(mix.index) ^ set(mix_std.index)) // 2, "of 25")

Raw features:          mean_sim spread = 0.090
Standardized features: mean_sim spread = 0.149
Songs that change between the two mixes: 10 of 25


## 总结：可以怎么讲

**原论文的结论，数据怎么说：**
- **结论 A 成立，而且比原来说的还明显。** danceability 区分派对歌的 AUC 比 0.5 还低，等于没用；派对分公式比单个特征好一点，但仍然很弱。原论文「不设阈值、改成排序」的决定是对的。
- **结论 B 要改得更准确。** 特征预测流派明显好于瞎猜，但也远谈不上可靠。更准确的说法是：音频特征**带有一些**流派信息，但不足以区分流派。

**复现时发现的方法问题：** 原始特征上的余弦相似度普遍很高（都在 0.88 以上），几乎每首歌都「很合群」。只是换成标准化后的特征，25 首里就有 10 首被换掉。也就是说，最终歌单很大程度取决于一个原论文没有讨论的预处理选择。

**局限：** 「派对歌」标签来自歌单名，很粗糙；流派也是歌单的流派，不一定是歌本身的流派（比如 *Believer* 被标成了 latin）。

**和研究方向的联系：** 表层的音频特征很难抓住音乐更深的属性（适不适合派对、属于什么流派）。这和「检测 AI 音乐需要找对特征」是同一个问题。